In [ ]:
import math
from datetime import datetime

def windowed_time_decay(doc_date_str, allowed_years, lambda_inside=0.005, lambda_outside=0.03):
    doc_date = datetime.strptime(doc_date_str, "%Y-%m-%d")
    now = datetime.utcnow()
    age_days = (now - doc_date).days
    cutoff_days = allowed_years * 365
    if age_days <= cutoff_days:
        return math.exp(-lambda_inside * age_days)
    else:
        return math.exp(-lambda_outside * age_days)


In [ ]:
def agentic_faiss_query(vectorstore, query_text, allowed_years=3, top_k=5, fetch_k=50):
    """
    - vectorstore: LangChain FAISS store
    - query_text: LLM-generated query
    - allowed_years: agent-specified temporal window
    - fetch_k: retrieve more candidates for re-ranking
    """
    # Step 1: semantic retrieval
    candidates = vectorstore.similarity_search(query_text, k=fetch_k)
    
    # Step 2: apply agent-guided temporal weighting
    results = []
    for doc in candidates:
        date_str = doc.metadata.get("date", None)
        if date_str:
            time_weight = windowed_time_decay(date_str, allowed_years)
        else:
            time_weight = 1.0  # no date -> no decay
        # FAISS inner product similarity score isn't directly exposed, but we can approximate
        # using 1.0 for now since we re-rank relatively
        final_score = time_weight
        results.append({"doc": doc, "final_score": final_score})
    
    # Step 3: re-rank
    results.sort(key=lambda x: x["final_score"], reverse=True)
    
    return [r["doc"] for r in results[:top_k]]


In [ ]:
PROMPT = (
    f"CURRENT CLINICAL SUMMARY:\n{state['template']}\n\n"
    f"PAST ACTIONS:\n{actionhistory2str(state['action_history'])}\n\n"

    "You are an autonomous clinical query agent building a patient's pre-visit clinical summary. "
    "Your role is to generate **one focused query** per turn that will be used to retrieve information from the hospital EHR, "
    "which includes structured data and physician notes (e.g., diagnoses, procedures, medications, labs, encounters, discharge summaries).\n\n"

    "Available actions:\n"
    "1. search_text – produce a query for clinical text or structured EHR data;\n"
    "2. search_imaging – produce a query for imaging impressions if clearly needed;\n"
    "3. finish – stop if the summary is clinically sufficient.\n\n"

    "Task: Identify the single most important missing clinical fact for physician decision-making "
    "and produce ONE focused action with a corresponding query.\n\n"

    "Rules: \n"
    "- Generate only one action and query per turn.\n"
    "- Do NOT repeat past actions listed above.\n"
    "- Prefer search_text over search_imaging unless imaging is essential.\n"
    "- Do NOT modify the clinical summary.\n"
    "- **Do not include time terms like 'recent' or 'last year' in the query text.** "
    "Use the optional 'allowed_years' parameter to indicate recency instead.\n\n"

    "Optional: If your query should focus on more recent data, include an 'allowed_years' field "
    "indicating how many years back to consider (e.g., recent labs or updated guidelines). "
    "If recency is not important, omit this field.\n\n"

    "Stop: choose 'finish' if key clinical information is complete, "
    "or prior searches added nothing useful. Do not generate additional actions after choosing finish.\n\n"

    "Output JSON only. Include 'allowed_years' only when relevant.\n"
    "Schema: {\"action\": <string>, \"query\": <string>, \"allowed_years\": <integer, optional>}\n\n"

    "Examples (each example returns only ONE action):\n"
    "1. {\"action\": \"search_text\", \"query\": \"current medication list with doses\", \"allowed_years\": 2}\n"
    "2. {\"action\": \"search_text\", \"query\": \"most recent HbA1c\", \"allowed_years\": 1}\n"
    "3. {\"action\": \"search_imaging\", \"query\": \"echocardiogram impression\", \"allowed_years\": 3}\n"
    "4. {\"action\": \"search_text\", \"query\": \"past hospital admissions for heart failure\"}\n"
    "5. {\"action\": \"finish\", \"query\": \"\"}"
)
